In [ ]:
# core Libs
import re
import numpy as np
import pandas as pd

# NLTK ---
import nltk

nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import WordPunctTokenizer
from nltk.stem import PorterStemmer, SnowballStemmer

# ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# visualization ---
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
data = pd.read_csv("C:/Users/Umar Dawood/Downloads/ai_ml_course_reviews.csv")


In [3]:
data.shape

(101, 2)

In [4]:
len(data)


101

In [5]:
data.columns

Index(['review_text', 'sentiment'], dtype='str')

In [6]:
data.dtypes

review_text      str
sentiment      int64
dtype: object

In [8]:
data.head(3)

,review_text,sentiment
0,The hands-on labs were genuinely well-designed...,1
1,I came in with zero ML background and by week ...,1
2,The instructor didn't just teach syntax — she ...,1


In [9]:
data['sentiment'].value_counts()

sentiment
1    51
0    50
Name: count, dtype: int64

In [10]:
# ============================================================
# TEXT PREPROCESSING PIPELINE
# 1. Lowercasing + Noise Removal
# 2. Tokenization
# 3. Stopword Removal
# 4. Stemming + Lemmatization
# 5. TF-IDF Vectorization
# ============================================================

# ----- Imports -----
import re
import numpy as np
import pandas as pd

# NLTK
import nltk
nltk.download("stopwords", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("averaged_perceptron_tagger", quiet=True)
nltk.download("averaged_perceptron_tagger_eng", quiet=True)

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize, WordPunctTokenizer
from nltk.stem import PorterStemmer, SnowballStemmer, WordNetLemmatizer

# sklearn
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

# ============================================================
# 1. LOWERCASING + NOISE REMOVAL
# ============================================================
def clean_text(text):
    """
    Comprehensive text cleaning:
    - Lowercase conversion
    - HTML tag removal
    - URL removal
    - Email removal
    - Special characters/punctuation removal
    - Extra whitespace removal
    """
    # Lowercase
    text = text.lower()
    
    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)
    
    # Remove URLs
    text = re.sub(r'http\S+|www\.\S+', '', text)
    
    # Remove email addresses
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '', text)
    
    # Remove special characters and punctuation (keep only letters and spaces)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Remove extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

# Apply cleaning
data['cleaned_text'] = data['review_text'].apply(clean_text)

print("="*80)
print("1. LOWERCASING + NOISE REMOVAL")
print("="*80)
for i in range(3):
    print(f"\nOriginal: {data['review_text'].iloc[i][:80]}...")
    print(f"Cleaned:  {data['cleaned_text'].iloc[i][:80]}...")

# ============================================================
# 2. TOKENIZATION
# ============================================================
sample_text = data['cleaned_text'].iloc[0]
print("\n" + "="*80)
print("2. TOKENIZATION")
print("="*80)
print(f"\nSample: {sample_text[:60]}...")

# Whitespace tokenization
whitespace_tokens = sample_text.split()
print(f"\nWhitespace tokens ({len(whitespace_tokens)}): {whitespace_tokens[:12]}")

# NLTK word_tokenize (linguistically-aware)
nltk_tokens = word_tokenize(sample_text)
print(f"\nNLTK word_tokenize ({len(nltk_tokens)}): {nltk_tokens[:12]}")

# WordPunctTokenizer
wp_tokenizer = WordPunctTokenizer()
wp_tokens = wp_tokenizer.tokenize(sample_text)
print(f"\nWordPunctTokenizer ({len(wp_tokens)}): {wp_tokens[:12]}")

print("\n--- WHY IT MATTERS ---")
print("- Whitespace: splits on spaces only, 'don't' = 1 token")
print("- NLTK tokenizer: correctly splits contractions, handles punctuation")
print("- Better tokenization → better feature extraction → better models")

# ============================================================
# 3. STOPWORD REMOVAL
# ============================================================
stop_words = set(stopwords.words('english'))
print("\n" + "="*80)
print("3. STOPWORD REMOVAL")
print("="*80)
print(f"\nTotal stopwords: {len(stop_words)}")
print(f"Sample: {list(stop_words)[:12]}")

# Keep negation words for sentiment
keep_words = {'not', 'no', 'never', 'nor', 'neither', 'none', 'nothing'}
custom_stopwords = stop_words - keep_words

def remove_stopwords_custom(text):
    tokens = word_tokenize(text)
    return [word for word in tokens if word.lower() not in custom_stopwords]

data['tokens_no_stop'] = data['cleaned_text'].apply(remove_stopwords_custom)
print(f"\nWith stopwords removed: {data['tokens_no_stop'].iloc[0][:12]}")

print("\n--- WHEN TO KEEP CERTAIN STOPWORDS ---")
print("- 'not', 'no', 'never' → Negation carries sentiment!")
print("- 'very', 'really', 'extremely' → Intensifiers")
print("- Example: 'not good' vs 'good' = opposite meanings")

# ============================================================
# 4. STEMMING + LEMMATIZATION
# ============================================================
porter_stemmer = PorterStemmer()
snowball_stemmer = SnowballStemmer("english")
wordnet_lemmatizer = WordNetLemmatizer()

test_words = ["running", "runs", "ran", "easily", "better", "best", 
              "programming", "programs", "went", "gone", "children", "feet"]

print("\n" + "="*80)
print("4. STEMMING + LEMMATIZATION")
print("="*80)
print(f"\n{'Original':<12} {'Porter':<12} {'Snowball':<12} {'Lemmatizer'}")
print("-"*50)

for word in test_words:
    porter = porter_stemmer.stem(word)
    snowball = snowball_stemmer.stem(word)
    lemma = wordnet_lemmatizer.lemmatize(word)
    print(f"{word:<12} {porter:<12} {snowball:<12} {lemma}")

print("\n--- TRADEOFF ANALYSIS ---")
print("- Stemming: Faster, simpler, but may produce non-words")
print("  → Porter: oldest, most common")
print("  → Snowball: improved, better quality")
print("- Lemmatization: Preserves meaning, produces real words")
print("  → Requires POS tagging for best results")
print("  → Slower but more accurate")

# Apply to sample
def apply_porter(tokens):
    return [porter_stemmer.stem(t) for t in tokens]
def apply_snowball(tokens):
    return [snowball_stemmer.stem(t) for t in tokens]
def apply_lemmatize(tokens):
    return [wordnet_lemmatizer.lemmatize(t, pos='v') for t in tokens]

sample_tokens = data['tokens_no_stop'].iloc[0][:10]
print(f"\nSample:     {sample_tokens}")
print(f"Porter:     {apply_porter(sample_tokens)}")
print(f"Snowball:   {apply_snowball(sample_tokens)}")
print(f"Lemmatized: {apply_lemmatize(sample_tokens)}")

# ============================================================
# 5. TF-IDF VECTORIZATION
# ============================================================
def full_preprocess(text, use_lemmatization=True):
    """Complete preprocessing pipeline"""
    text = clean_text(text)
    tokens = word_tokenize(text)
    tokens = [t for t in tokens if t.lower() not in custom_stopwords]
    if use_lemmatization:
        tokens = [wordnet_lemmatizer.lemmatize(t, pos='v') for t in tokens]
    else:
        tokens = [snowball_stemmer.stem(t) for t in tokens]
    return ' '.join(tokens)

data['preprocessed_text'] = data['review_text'].apply(lambda x: full_preprocess(x, True))

print("\n" + "="*80)
print("5. TF-IDF VECTORIZATION")
print("="*80)
print("\nPreprocessed samples:")
for i in range(2):
    print(f"\nOriginal:     {data['review_text'].iloc[i][:70]}...")
    print(f"Preprocessed: {data['preprocessed_text'].iloc[i][:70]}...")

# TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),     # Unigrams + Bigrams
    max_features=1000,      # Limit vocabulary
    min_df=2,               # Min document frequency
    max_df=0.95,            # Max document frequency
    stop_words='english'
)

X = tfidf_vectorizer.fit_transform(data['preprocessed_text'])
y = data['sentiment']

print("\n--- TF-IDF CONFIGURATION ---")
print(f"- N-grams: (1,2) unigrams + bigrams")
print(f"- Max features: 1000")
print(f"- Min DF: 2, Max DF: 95%")
print(f"\nTF-IDF Matrix shape: {X.shape}")
print(f"Vocabulary size: {len(tfidf_vectorizer.vocabulary_)}")

feature_names = tfidf_vectorizer.get_feature_names_out()
print(f"\nSample features: {feature_names[:15].tolist()}")

print("\n" + "="*80)
print("PREPROCESSING COMPLETE!")
print("="*80)

1. LOWERCASING + NOISE REMOVAL

Original: The hands-on labs were genuinely well-designed — I left each session with workin...
Cleaned:  the handson labs were genuinely welldesigned i left each session with working co...

Original: I came in with zero ML background and by week six I was building end-to-end pipe...
Cleaned:  i came in with zero ml background and by week six i was building endtoend pipeli...

Original: The instructor didn't just teach syntax — she kept connecting algorithms back to...
Cleaned:  the instructor didnt just teach syntax she kept connecting algorithms back to re...

2. TOKENIZATION

Sample: the handson labs were genuinely welldesigned i left each ses...

Whitespace tokens (19): ['the', 'handson', 'labs', 'were', 'genuinely', 'welldesigned', 'i', 'left', 'each', 'session', 'with', 'working']

NLTK word_tokenize (19): ['the', 'handson', 'labs', 'were', 'genuinely', 'welldesigned', 'i', 'left', 'each', 'session', 'with', 'working']

WordPunctTokenizer (19): ['the